# Group12 Hex AI - Neural Network Training
## AlphaZero-Style Training on Google Colab GPU

**Author**: Group12  
**Course**: COMP34111 AI & Games  
**Goal**: Train championship-level Hex AI using neural networks + MCTS  
**Expected Training Time**: 2-4 hours on T4 GPU  
**Expected Result**: 85-95% win rate vs heuristic agents

---

## 📋 Training Configuration

- **Iterations**: 10 (AlphaZero cycles)
- **Games per iteration**: 1000 (self-play games)
- **MCTS simulations**: 800 per move
- **Network**: ResNet (10 blocks, 256 channels)
- **Board size**: 11×11
- **Total games**: ~10,000

---

## 🚀 Quick Start

**Run all cells in order**. The notebook will:
1. ✅ Setup GPU environment
2. ✅ Upload your codebase
3. ✅ Install dependencies
4. ✅ Run training (2-4 hours)
5. ✅ Download trained model

**💾 IMPORTANT**: Save checkpoints every 30 minutes (Colab has 12-hour limit)


## Step 1: Verify GPU Access

Ensure you're using GPU runtime:
- **Runtime** → **Change runtime type** → **GPU** (T4, V100, or A100)


In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU")

## Step 2: Upload Codebase

**Option A**: Upload ZIP file (recommended for first time)
**Option B**: Clone from GitHub (if you have a private repo)

### Option A: Upload ZIP


In [ ]:
# Upload your COMP34111-AI-Games-Hex folder as a ZIP file
from google.colab import files
import zipfile
import os

print("Please upload your COMP34111-AI-Games-Hex.zip file:")
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("✅ Extraction complete!")

# Smart directory detection (handles both subdirectory and flat extraction)
project_dir = None

# Check for subdirectory first (proper structure)
dirs = [d for d in os.listdir('/content/')
        if os.path.isdir(f'/content/{d}') and ('COMP34111' in d or 'AI-Games' in d)]
if dirs:
    project_dir = f"/content/{dirs[0]}"
    print(f"✅ Found subdirectory: {project_dir}")
# Check if files extracted directly to /content/ (flat structure)
elif os.path.exists('/content/Hex.py') and os.path.exists('/content/agents'):
    project_dir = '/content'
    print(f"✅ Files extracted directly to: {project_dir}")
else:
    print("⚠️  Could not find project files!")
    print(f"Contents of /content/: {os.listdir('/content/')}")

# Change to project directory
if project_dir:
    os.chdir(project_dir)
    print(f"✅ Working directory: {os.getcwd()}")
    # Verify key files
    if os.path.exists('Hex.py') and os.path.exists('agents/Group12'):
        print("✅ Project files verified - ready to proceed!")
    else:
        print("⚠️  WARNING: Some project files may be missing")
else:
    print("❌ ERROR: Could not set working directory")

### Option B: Clone from GitHub (Alternative)


In [ ]:
# Uncomment and modify if using GitHub:
# !git clone https://github.com/YOUR_USERNAME/COMP34111-AI-Games-Hex.git
# %cd COMP34111-AI-Games-Hex

## Step 3: Install Dependencies


In [ ]:
# Install required packages
# PyTorch is already installed in Colab with CUDA support

!pip install tensorboard tqdm numpy scipy

print("\n✅ Dependencies installed!")

## Step 4: Verify Training Code


In [ ]:
# Verify critical files exist
import os

required_files = [
    'agents/Group12/training/trainer.py',
    'agents/Group12/neural/hex_network.py',
    'agents/Group12/neural/board_encoder.py',
    'agents/Group12/Group12Agent_neural.py',
]

all_found = True
for file in required_files:
    if os.path.exists(file):
        print(f"✅ {file}")
    else:
        print(f"❌ {file} NOT FOUND")
        all_found = False

if all_found:
    print("\n✅ All required files found! Ready to train.")
else:
    print("\n⚠️  Missing files. Please check your upload.")

## Step 5: Configure Training Parameters

You can adjust these based on time constraints:

- **Quick test** (30 min): `iterations=2, games=100`
- **Standard** (2-4 hours): `iterations=10, games=1000` ← **RECOMMENDED**
- **Full training** (8-12 hours): `iterations=20, games=2000`


In [ ]:
# Training configuration - OPTIMIZED for better learning
# Key changes:
# - Reduced games per iteration (faster iteration cycles)
# - Reduced epochs (prevents overfitting)
# - Reduced simulations (faster games, still effective)

TRAINING_CONFIG = {
    'num_iterations': 10,          # AlphaZero iterations
    'games_per_iteration': 100,    # Reduced from 1000 (faster cycles, more iterations)
    'epochs_per_iteration': 5,     # Reduced from 10 (prevents overfitting!)
    'batch_size': 256,             # Batch size for training
    'num_simulations': 100,        # Reduced from 800 (60s/game instead of 7min/game)
    'learning_rate': 0.001,        # Initial learning rate
    'model_dir': 'models',         # Where to save models
    'log_dir': 'runs',             # TensorBoard logs
}

print("Training Configuration (OPTIMIZED):")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

# Estimate training time with optimized settings
total_games = TRAINING_CONFIG['num_iterations'] * TRAINING_CONFIG['games_per_iteration']
games_per_minute = 1.0  # ~60s per game with 100 simulations
est_time_hours = (total_games / games_per_minute) / 60
print(f"\nEstimated training time: {est_time_hours:.1f} - {est_time_hours * 1.5:.1f} hours")
print(f"Total self-play games: {total_games:,}")
print("\n--- KEY IMPROVEMENTS ---")
print("1. Dynamic temperature schedule (0.5-1.5, never drops to 0)")
print("2. Label smoothing (0.1) prevents overconfident predictions")
print("3. Gradient clipping (max_norm=1.0) prevents exploding gradients")
print("4. L2 regularization (1e-4) prevents overfitting")
print("5. Faster iteration cycles = more learning opportunities")

## Step 6: Run Training

**⚠️  IMPORTANT**:
- This will take 2-4 hours
- Colab sessions timeout after 12 hours of inactivity
- Models are saved every iteration (checkpoint every ~20 min)
- Keep this tab open to monitor progress


In [ ]:
# Create training script
import sys
sys.path.insert(0, os.getcwd())

from agents.Group12.training.trainer import HexTrainer
from agents.Group12.neural.hex_network import HexNeuralNetwork
from datetime import datetime

print("="*70)
print("GROUP12 HEX AI - NEURAL NETWORK TRAINING")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print("="*70)
print()

# Create network
network = HexNeuralNetwork(
    board_size=11,
    num_res_blocks=10,
    num_channels=256
)

# Create trainer
trainer = HexTrainer(
    network=network,
    learning_rate=TRAINING_CONFIG['learning_rate'],
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"Network parameters: {sum(p.numel() for p in network.parameters()):,}")
print()

# Run training
try:
    trainer.train(
        num_iterations=TRAINING_CONFIG['num_iterations'],
        games_per_iteration=TRAINING_CONFIG['games_per_iteration'],
        epochs_per_iteration=TRAINING_CONFIG['epochs_per_iteration'],
        batch_size=TRAINING_CONFIG['batch_size'],
        num_simulations=TRAINING_CONFIG['num_simulations'],
        model_dir=TRAINING_CONFIG['model_dir'],
        log_dir=TRAINING_CONFIG['log_dir']
    )
    
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print("="*70)
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Models saved in: {TRAINING_CONFIG['model_dir']}/")
    print("="*70)
    
except KeyboardInterrupt:
    print("\n⚠️  Training interrupted by user")
    print(f"Partial models saved in: {TRAINING_CONFIG['model_dir']}/")
    
except Exception as e:
    print(f"\n❌ Error during training: {e}")
    import traceback
    traceback.print_exc()

## Step 7: Monitor Training (Optional)

Run TensorBoard to visualize training progress:


In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir runs

# You can view:
# - Policy loss (should decrease)
# - Value loss (should decrease)
# - Win rate (should increase)

## Step 8: Quick Evaluation

Test the trained model against a simple agent:


In [ ]:
# Quick validation test
import os

# Find the best model
model_files = [f for f in os.listdir(TRAINING_CONFIG['model_dir']) if f.endswith('.pth')]
if model_files:
    best_model = sorted(model_files)[-1]  # Get latest
    print(f"Best model: {best_model}")
    print(f"Model size: {os.path.getsize(f"{TRAINING_CONFIG['model_dir']}/{best_model}") / 1e6:.2f} MB")
    
    # Test game (optional)
    print("\n🎮 Running quick test game...")
    !python3 Hex.py \
        -p1 "agents.Group12.Group12Agent_neural Group12Agent" \
        -p2 "agents.Group12.Group12Agent_simple Group12Agent" \
        -v
else:
    print("⚠️  No model files found")

## Step 9: Download Trained Models

Download the trained models to your local machine:


In [ ]:
# Zip models directory
import shutil
from google.colab import files

# Create ZIP
shutil.make_archive('trained_models', 'zip', TRAINING_CONFIG['model_dir'])
print("✅ Models zipped")

# Download
print("Downloading trained_models.zip...")
files.download('trained_models.zip')

print("\n📦 Download complete!")
print("Extract this ZIP to your project's models/ directory")

## Step 10: Backup to Google Drive (Optional)

Save models to your Google Drive for safety:


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy models to Drive
import shutil
drive_path = '/content/drive/MyDrive/Group12_Hex_Models'
!mkdir -p {drive_path}
!cp -r {TRAINING_CONFIG['model_dir']}/* {drive_path}/

print(f"✅ Models backed up to Google Drive: {drive_path}")

## 🎯 Next Steps

After downloading the trained model:

1. **Extract** `trained_models.zip` to your project's `models/` directory

2. **Update cmd.txt** to use neural agent:
   ```bash
   echo "agents.Group12.Group12Agent_neural Group12Agent" > agents/Group12/cmd.txt
   ```

3. **Test locally**:
   ```bash
   python3 Hex.py -p1 "agents.Group12.Group12Agent_neural Group12Agent" \
                  -p2 "agents.Group12.Group12Agent Group12Agent"
   ```

4. **Run comprehensive tests** (50 games):
   ```bash
   python3 run_comprehensive_tests.py --games 50 --output results_neural_vs_full.json
   ```

5. **Expected results**:
   - Neural agent should win 85-95% vs Full agent
   - If win rate < 70%, try training longer
   - If win rate > 90%, ready for tournament!

---

## 📊 Training Statistics

Track your training runs:

| Run | Date | Iterations | Games | Time | Best Win Rate | Notes |
|-----|------|-----------|-------|------|---------------|-------|
| 1   | -    | 10        | 10k   | -    | -             | -     |

---

## 🐛 Troubleshooting

**Issue**: Training is too slow
- **Solution**: Reduce `games_per_iteration` to 500
- **Solution**: Reduce `num_simulations` to 400

**Issue**: Out of memory error
- **Solution**: Reduce `batch_size` to 128
- **Solution**: Reduce `num_channels` to 128

**Issue**: Colab disconnected
- **Solution**: Models are saved every iteration in `models/`
- **Solution**: Resume from latest checkpoint

**Issue**: Model not learning (win rate not improving)
- **Solution**: Train longer (20+ iterations)
- **Solution**: Increase `num_simulations` to 1600

---

## 📚 References

- AlphaZero paper: https://arxiv.org/abs/1712.01815
- Hex game theory: PROJECT_STATUS_NOVEMBER_2024.md
- Training methodology: agents/Group12/training/trainer.py

---

**Good luck with training! 🚀**
